# Test PaperQA with Three Locally-Hosted NIM APIs

Prerequisites:
- Launch NIMs -> `launch_NIMs.sh`
- Install PaperQA -> `install_PQA.sh`

***Use the Jupyter Kernel built in last step.***

In [1]:
import logging
import sys

logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s")
for _name in ("LiteLLM", "litellm"):
    _log = logging.getLogger(_name)
    _log.setLevel(logging.INFO)
    if not _log.handlers:
        _h = logging.StreamHandler(sys.stdout)
        _h.setLevel(logging.INFO)
        _h.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
        _log.addHandler(_h)
    _log.propagate = False

In [2]:
# # Test if paperqa_nemotron is available
import paperqa
print(f"PaperQA location: {paperqa.__file__}")

# Log embedding call
from paperqa.llms import LiteLLMEmbeddingModel
_embed_log = logging.getLogger("LiteLLM")
_orig_embed = LiteLLMEmbeddingModel.embed_documents
async def _logged_embed(self, *args, **kwargs):
    texts = args[0] if args else kwargs.get("texts", [])
    n = len(texts) if texts else 0
    kind = "query" if n == 1 else "embed_documents"
    msg = f"LiteLLM embedding() model= {getattr(self, 'name', '?')}; ({kind}) n={n}"
    _embed_log.info(msg)
    if not _embed_log.handlers:
        print(f"INFO: {msg}")
    return await _orig_embed(self, *args, **kwargs)
LiteLLMEmbeddingModel.embed_documents = _logged_embed

try:
    import paperqa_nemotron
    from paperqa_nemotron import parse_pdf_to_pages
    print(f"✅ paperqa_nemotron location: {paperqa_nemotron.__file__}")
    print(f"✅ parse_pdf_to_pages available: {parse_pdf_to_pages}")
except ImportError as e:
    print(f"❌ paperqa_nemotron NOT installed!")
    print(f"   Error: {e}")
    print(f"   Run: pip install -e '.[local,pymupdf,nemotron]'")

04:43:31 - LiteLLM:WARNING: get_model_cost_map.py:174 - LiteLLM: Failed to fetch remote model cost map from : Request URL is missing an 'http://' or 'https://' protocol.. Falling back to local backup.


PaperQA location: /home/shadeform/paper-qa/src/paperqa/__init__.py
✅ paperqa_nemotron location: /home/shadeform/paper-qa/.venv/lib/python3.11/site-packages/paperqa_nemotron/__init__.py
✅ parse_pdf_to_pages available: <function parse_pdf_to_pages at 0x7c66c2ea9a80>


## Step 1: Load PDF Papers (multiple)

In [3]:
# Create papers directory
!mkdir -p papers

**Upload multiple PDFs to the `papers/` folder, then run the cell below to auto-detect all of them:**

In [4]:
import os
import glob

# Auto-detect PDF files in the papers folder
pdf_files = sorted(glob.glob("papers/*.pdf"))

if pdf_files:
    print(f"📁 Found {len(pdf_files)} PDF(s) in papers/ (all will be added):")
    for i, pdf in enumerate(pdf_files):
        size_mb = os.path.getsize(pdf) / (1024 * 1024)
        print(f"   [{i}] {pdf} ({size_mb:.2f} MB)")
else:
    print("❌ No PDFs found in papers/ folder!")
    print("   Upload PDFs to papers/ and re-run this cell.")
    pdf_files = []

📁 Found 2 PDF(s) in papers/ (all will be added):
   [0] papers/Acid-sensing ion channels 1a (ASIC1a) inhibit ne_600b016999c9043b.pdf (1.92 MB)
   [1] papers/attention_is_all_you_need.pdf (2.11 MB)


## Step 2: Configure Locally-Hosted Endpoints

We configure three locally-hosted NIMs (started via `launch_NIMs.sh`):

- **Parse** nvidia/nemotron-parse -> PDF parser (localhost:8002)
- **Embedding** nvidia/llama-3.2-nv-embedqa-1b-v2 -> embedding (localhost:8003)
- **VLM** nvidia/nemotron-nano-12b-v2-vl -> enrichment_llm, summary_llm, llm, agent_llm (localhost:8004)
  
*Right-side terms match paper-qa `Settings` / `ParsingSettings` field names.*

In [5]:
# =============================================================================
# NIM CONFIGURATION (Parse=8002, Embedding=8003, VLM=8004)
# =============================================================================
import os
SELFHOST_API_KEY = "dummy"

# ----- Parse -----
PARSE_API_BASE = "http://localhost:8002/v1"
PARSE_API_KEY = "dummy"
PARSE_MODEL_NAME = "nvidia/nemotron-parse"
PARSE_COMPLETION_KWARGS = {
    "temperature": 0,
    "max_tokens": 8995,
}

# ----- Embedding --------------------
EMBEDDING_API_BASE = "http://localhost:8003/v1"
EMBEDDING_MODEL = "nvidia/llama-3.2-nv-embedqa-1b-v2"

# ----- VLM --------------------
VLM_API_BASE = "http://localhost:8004/v1"
VLM_MODEL = "nvidia/nemotron-nano-12b-v2-vl"
CUSTOM_VLM_NAME = "selfhost-nemotron-vlm"

print("\n=== Locally-Hosted NIM Endpoints ===")
print(f"Nemotron-Parse: {PARSE_API_BASE} ({PARSE_MODEL_NAME})")
print(f"Embedding: {EMBEDDING_API_BASE}  ({EMBEDDING_MODEL})")
print(f"VLM: {VLM_API_BASE}  ({VLM_MODEL})")


=== Locally-Hosted NIM Endpoints ===
Nemotron-Parse: http://localhost:8002/v1 (nvidia/nemotron-parse)
Embedding: http://localhost:8003/v1  (nvidia/llama-3.2-nv-embedqa-1b-v2)
VLM: http://localhost:8004/v1  (nvidia/nemotron-nano-12b-v2-vl)


## Step 3: Create PaperQA Settings

In [6]:
from paperqa import Settings
from paperqa.settings import AgentSettings, IndexSettings, AnswerSettings, ParsingSettings
from paperqa_nemotron import parse_pdf_to_pages
import pathlib

# VLM config for enrichment_llm, summary_llm, llm, agent_llm (locally-hosted nemotron-nano-12b-v2-vl on 8004)
nvidia_vlm_config = {
    "model_list": [
        {
            "model_name": CUSTOM_VLM_NAME,
            "litellm_params": {
                "model": f"openai/{VLM_MODEL}",
                "api_base": VLM_API_BASE,
                "api_key": SELFHOST_API_KEY,
                "temperature": 0,
                "max_tokens": 2048,
            },
        }
    ]
}

# Embedding config (locally-hosted llama-3.2-nv-embedqa-1b-v2 on 8003)
nvidia_embedding_config = {
    "kwargs": {
        "api_base": EMBEDDING_API_BASE,
        "api_key": SELFHOST_API_KEY,
        "encoding_format": "float",
        "input_type": "passage",     
    }
}

# ParsingSettings with Nemotron-Parse NIM
parsing_settings = ParsingSettings(
    # Avoid Semantic Scholar / metadata API rate limits (429)
    use_doc_details=False,
    # Use nemotron-parse as the PDF parser
    parse_pdf=parse_pdf_to_pages,
    
    # reader_config is passed to parse_pdf_to_pages(); api_params go to LiteLLM
    reader_config={
        "chunk_chars": 5000,
        "overlap": 250,
        "dpi": 150,  # Image resolution for PDF rendering
        
        "api_params": {
            "api_base": PARSE_API_BASE,       
            "api_key": PARSE_API_KEY,         
            "model_name": PARSE_MODEL_NAME, 
            **PARSE_COMPLETION_KWARGS, 
        }
    },
    
    # Enrichment LLM for multimodal (self-hosted VLM on 8004)
    enrichment_llm=CUSTOM_VLM_NAME,
    enrichment_llm_config=nvidia_vlm_config,
    multimodal=True,  # Enable multimodal to use nemotron-parse's full capabilities
)

# Full settings
settings = Settings(
    # LLMs
    llm=CUSTOM_VLM_NAME,
    llm_config=nvidia_vlm_config,
    summary_llm=CUSTOM_VLM_NAME,
    summary_llm_config=nvidia_vlm_config,
    
    # Embedding (self-hosted on 8003)
    embedding=f"openai/{EMBEDDING_MODEL}",
    embedding_config=nvidia_embedding_config,
    
    # Temperature for all LLMs (answer, summary, agent, enrichment) unless overridden in their config
    temperature=0,
    # Global logging level 0-3 for LLM/embedding calls (not per-model)
    verbosity=3,
    
    answer=AnswerSettings(
        evidence_k=5,
        answer_max_sources=3,
    ),
    
    # Use our nemotron-parse settings
    parsing=parsing_settings,
    
    agent=AgentSettings(
        agent_llm=CUSTOM_VLM_NAME,
        agent_llm_config=nvidia_vlm_config,
        index=IndexSettings(
            paper_directory=pathlib.Path.cwd() / "papers",
        ),
    ),
)

print("✅ PaperQA Settings created!")
print(f"   PDF Parser: {parsing_settings.parse_pdf}")
print(f"   Nemotron Parse API Base: {parsing_settings.reader_config['api_params']['api_base']}")
print(f"   Multimodal: {parsing_settings.multimodal}")
print(f"   Embedding: {settings.embedding}")
print(f"   LLM: {settings.llm}")

✅ PaperQA Settings created!
   PDF Parser: <function parse_pdf_to_pages at 0x7c66c2ea9a80>
   Nemotron Parse API Base: http://localhost:8002/v1
   Multimodal: True
   Embedding: openai/nvidia/llama-3.2-nv-embedqa-1b-v2
   LLM: selfhost-nemotron-vlm


## Step 4: Add All Papers with Nemotron-Parse

In [7]:
from paperqa import Docs

# Create a Docs object and add all PDFs
docs = Docs()

if not pdf_files:
    print("❌ No PDFs to add. Run the cell above to detect PDFs in papers/.")
else:
    print(f"Adding {len(pdf_files)} paper(s) to Docs (using nemotron-parse)...")
    added = 0
    for i, path in enumerate(pdf_files):
        try:
            name = await docs.aadd(path, settings=settings)
            if name:
                added += 1
                print(f"   [{i+1}/{len(pdf_files)}] ✅ {path}")
            else:
                print(f"   [{i+1}/{len(pdf_files)}] ⚠️ Skipped (already in collection): {path}")
        except Exception as e:
            print(f"   [{i+1}/{len(pdf_files)}] ❌ Failed: {path}")
            print(f"      Error: {e}")
            import traceback
            traceback.print_exc()
    print(f"\n✅ Added {added} doc(s). Total docs: {len(docs.docs)}")
    for doc_key, doc in docs.docs.items():
        print(f"   - {doc.docname}: {doc.citation[:80]}...")

Adding 2 paper(s) to Docs (using nemotron-parse)...
INFO: 
LiteLLM completion() model= nvidia/nemotron-parse; provider = custom_openai


04:43:44 - LiteLLM:INFO: utils.py:3889 - 
LiteLLM completion() model= nvidia/nemotron-parse; provider = custom_openai


INFO: 
LiteLLM completion() model= nvidia/nemotron-parse; provider = custom_openai


04:43:44 - LiteLLM:INFO: utils.py:3889 - 
LiteLLM completion() model= nvidia/nemotron-parse; provider = custom_openai


INFO: 
LiteLLM completion() model= nvidia/nemotron-parse; provider = custom_openai


...

INFO: 
LiteLLM completion() model= nvidia/nemotron-nano-12b-v2-vl; provider = openai


04:44:46 - LiteLLM:INFO: utils.py:3889 - 
LiteLLM completion() model= nvidia/nemotron-nano-12b-v2-vl; provider = openai
04:44:46 - LiteLLM:INFO: utils.py:3889 - 
LiteLLM completion() model= nvidia/nemotron-nano-12b-v2-vl; provider = openai
04:44:46 - LiteLLM:INFO: utils.py:3889 - 
LiteLLM completion() model= nvidia/nemotron-nano-12b-v2-vl; provider = openai


INFO: LiteLLM embedding() model= openai/nvidia/llama-3.2-nv-embedqa-1b-v2; (embed_documents) n=9


04:44:55 - LiteLLM:INFO: 1319527074.py:14 - LiteLLM embedding() model= openai/nvidia/llama-3.2-nv-embedqa-1b-v2; (embed_documents) n=9


   [2/2] ✅ papers/attention_is_all_you_need.pdf

✅ Added 2 doc(s). Total docs: 2
   - urbano2014acidsensingionchannels: Francisco J. Urbano, Noelia G. Lino, Carlota M. F. González-Inchauspe, Laura E. ...
   - Vaswani2017: Vaswani, Ashish, et al. "Attention Is All You Need." *Advances in Neural Informa...


## Step 5: `docs.aquery()` -> non-agent, no tools

In [8]:
# Query the docs
print("Querying docs...")

try:
    session = await docs.aquery(
        "What experiments are carried out?",
        settings=settings,
    )
    
    print("✅ Query SUCCESS!")
    print("\n" + "=" * 60)
    print("Question:")
    print("=" * 60)
    print(session.question)
    
    print("\n" + "=" * 60)
    print("Answer:")
    print("=" * 60)
    print(session.answer)
    
    print("\n" + "=" * 60)
    print("References:")
    print("=" * 60)
    print(session.references)
    
except Exception as e:
    print(f"❌ Query FAILED!")
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

Querying docs...
INFO: LiteLLM embedding() model= openai/nvidia/llama-3.2-nv-embedqa-1b-v2; (query) n=1


04:45:18 - LiteLLM:INFO: 1319527074.py:14 - LiteLLM embedding() model= openai/nvidia/llama-3.2-nv-embedqa-1b-v2; (query) n=1


INFO: 
LiteLLM completion() model= nvidia/nemotron-nano-12b-v2-vl; provider = openai


04:45:18 - LiteLLM:INFO: utils.py:3889 - 
LiteLLM completion() model= nvidia/nemotron-nano-12b-v2-vl; provider = openai


INFO: 
LiteLLM completion() model= nvidia/nemotron-nano-12b-v2-vl; provider = openai


...

INFO: 
LiteLLM completion() model= nvidia/nemotron-nano-12b-v2-vl; provider = openai


04:45:21 - LiteLLM:INFO: utils.py:3889 - 
LiteLLM completion() model= nvidia/nemotron-nano-12b-v2-vl; provider = openai


INFO: 
LiteLLM completion() model= nvidia/nemotron-nano-12b-v2-vl; provider = openai


04:45:23 - LiteLLM:INFO: utils.py:3889 - 
LiteLLM completion() model= nvidia/nemotron-nano-12b-v2-vl; provider = openai


✅ Query SUCCESS!

Question:
What experiments are carried out?

Answer:
The experiments described in the context involve training and evaluating Transformer models for machine translation tasks (English-to-German and English-to-French) on the WMT 2014 dataset (Vaswani2017 pages 8-9). Variations in model architecture, including the number of attention heads, attention key/value dimensions, dropout rates, and positional encoding methods, were tested. Additionally, the Transformer was applied to English constituency parsing on the Penn Treebank dataset (Vaswani2017 pages 8-9). Results demonstrated that the Transformer (big) model achieved state-of-the-art BLEU scores of 28.4 and 41.8 for EN-DE and EN-FR, respectively, with significantly lower training costs compared to previous models (Vaswani2017 pages 8-9).  

Separately, experiments on acid-sensing ion channel 1a (ASIC1a) in female mice were conducted to investigate its role in neuromuscular transmission. These included electrophysiolog

## Step 6 `agent_query()` -> agent, tools


In [9]:
from paperqa.agents.main import agent_query

agent_question = "What experiments are carried out?"
print(f"Running agent_query with docs (pre-loaded papers)...")
print(f"Question: {agent_question}\n")

try:
    response = await agent_query(agent_question, settings, docs=docs)
    print("✅ agent_query SUCCESS!")
    print(f"Status: {response.status}")
    print("\n" + "=" * 60)
    print("Answer:")
    print("=" * 60)
    print(response.session.answer or "(no answer)")
except Exception as e:
    print(f"❌ agent_query failed: {e}")
    import traceback
    traceback.print_exc()

Running agent_query with docs (pre-loaded papers)...
Question: What experiments are carried out?

INFO: 
LiteLLM completion() model= nvidia/nemotron-parse; provider = custom_openai


04:46:40 - LiteLLM:INFO: utils.py:3889 - 
LiteLLM completion() model= nvidia/nemotron-parse; provider = custom_openai


INFO: 
LiteLLM completion() model= nvidia/nemotron-parse; provider = custom_openai


04:46:40 - LiteLLM:INFO: utils.py:3889 - 
LiteLLM completion() model= nvidia/nemotron-parse; provider = custom_openai


INFO: 
LiteLLM completion() model= nvidia/nemotron-parse; provider = custom_openai


...

INFO: 
LiteLLM completion() model= nvidia/nemotron-nano-12b-v2-vl; provider = openai


04:47:26 - LiteLLM:INFO: utils.py:3889 - 
LiteLLM completion() model= nvidia/nemotron-nano-12b-v2-vl; provider = openai


[agent LLM request] step=4 length=3693 chars
[agent LLM request] body:
[
  {
    "role": "system",
    "content": "You are a helpful AI assistant."
  },
  {
    "role": "user",
    "content": "Use the tools to answer the question: What experiments are carried out?\n\nWhen the answer looks sufficient, you can terminate by calling the complete tool. If the answer does not look sufficient, and you have already tried to answer several times with different evidence, terminate by calling the complete tool. The current status of evidence/papers/cost is Status: Paper Count=0 | Relevant Papers=0 | Current Evidence=0 | Current Cost=$0.0000"
  },
  {
    "role": "assistant",
    "content": "",
    "tool_calls": [
      {
        "id": "chatcmpl-tool-a39bca2b9f2740f68bcff1b198ebd997",
        "type": "function",
        "function": {
          "arguments": "{\"query\": \"experiments carried out\", \"min_year\": null, \"max_year\": null}",
          "name": "paper_search"
        }
      }
    ]
  

04:47:29 - LiteLLM:INFO: utils.py:3889 - 
LiteLLM completion() model= nvidia/nemotron-nano-12b-v2-vl; provider = openai


✅ agent_query SUCCESS!
Status: success

Answer:
The experiments include training Transformer models for English-to-German and English-to-French translation on the WMT 2014 dataset, with comparisons of BLEU scores and training costs against prior models. Variations in model architecture, such as the number of attention heads and dropout rates, are tested to assess their impact on performance (Vaswani2017 pages 8-9). Additionally, the Transformer architecture is applied to English constituency parsing on the Penn Treebank dataset (Vaswani2017 pages 8-9).  

In the study of ASIC1a's role in neuromuscular transmission, electrophysiological recordings are used to measure synaptic responses, while electron microscopy analyzes neuromuscular junction (NMJ) morphology. Functional assays evaluate the effects of ASIC1a on neurotransmitter release and muscle fatigue, with comparisons between male and female ASIC3-/- mice to explore sex-specific effects (urbano2014acidsensingionchannels pages 10-10

## Step 6.5: `ask()` -> agent, tools

Test the full agent workflow:

In [10]:
from paperqa import ask

print("Running agent-based query with ask()...")
print("This will search, gather evidence, and generate an answer.\n")

try:
    response = await ask(
        "What experiments are carried out?",
        settings=settings,
    )
    
    print("✅ Agent query SUCCESS!")
    print(f"\n--- Status: {response.status} ---")
    
    print("\n" + "=" * 60)
    print("Question:")
    print("=" * 60)
    print(response.session.question)
    
    print("\n" + "=" * 60)
    print("Answer:")
    print("=" * 60)
    print(response.session.answer)
    
    print("\n" + "=" * 60)
    print("Contexts Used (top 3):")
    print("=" * 60)
    for i, ctx in enumerate(response.session.contexts[:3], 1):
        print(f"\nContext {i} (score: {ctx.score}):")
        print(f"  Source: {ctx.text.name}")
        print(f"  Summary: {ctx.context[:200]}...")
        
except Exception as e:
    print(f"❌ Agent query FAILED!")
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

Running agent-based query with ask()...
This will search, gather evidence, and generate an answer.

PaperQA version: 0.1.dev906+g1674e0958


[04:50:27] Beginning agent 'ToolSelector' run with question 'What experiments are carried out?' and full settings  
           {'llm': 'selfhost-nemotron-vlm', 'llm_config': {'model_list': [{'model_name': 'selfhost-nemotron-vlm',  
           'litellm_params': {'model': 'openai/nvidia/nemotron-nano-12b-v2-vl', 'api_base':                        
           'http://localhost:8004/v1', 'api_key': 'dummy', 'temperature': 0, 'max_tokens': 2048}}]}, 'summary_llm':
           'selfhost-nemotron-vlm', 'summary_llm_config': {'model_list': [{'model_name': 'selfhost-nemotron-vlm',  
           'litellm_params': {'model': 'openai/nvidia/nemotron-nano-12b-v2-vl', 'api_base':                        
           'http://localhost:8004/v1', 'api_key': 'dummy', 'temperature': 0, 'max_tokens': 2048}}]}, 'embedding':  
           'openai/nvidia/llama-3.2-nv-embedqa-1b-v2', 'embedding_config': {'kwargs': {'api_base':                 
           'http://localhost:8003/v1', 'api_key': 'dummy', 'encoding_format': 'float', 'input_type': 'passage'}},  
           'temperature': 0.0, 'batch_size': 1, 'texts_index_mmr_lambda': 1.0, 'verbosity': 3, 'answer':           
           {'evidence_k': 5, 'evidence_retrieval': True, 'evidence_relevance_score_cutoff': 1,                     
           'evidence_summary_length': 'about 100 words', 'evidence_skip_summary': False,                           
           'evidence_text_only_fallback': False, 'answer_max_sources': 3, 'max_answer_attempts': None,             
           'answer_length': 'about 200 words, but can be longer', 'max_concurrent_requests': 4,                    
           'answer_filter_extra_background': False, 'get_evidence_if_no_contexts': True,                           
           'group_contexts_by_question': False, 'skip_evidence_citation_strip': False}, 'parsing':                 
           {'page_size_limit': 1280000, 'use_doc_details': True, 'reader_config': {'chunk_chars': 5000, 'overlap': 
           250, 'dpi': 150, 'api_params': {'api_base': 'http://localhost:8002/v1', 'api_key': 'dummy',             
           'model_name': 'nvidia/nemotron-parse', 'temperature': 0, 'max_tokens': 8995}}, 'multimodal': True,      
           'citation_prompt': "Provide the citation for the following text in MLA Format. Do not write an          
           introductory sentence. Do not fabricate a DOI such as '10.xxxx' if one cannot be found, just leave it   
           out of the citation. If reporting date accessed, the current year is 2026\n\n{text}\n\nCitation:",      
           'structured_citation_prompt': "Extract the title, authors, and doi as a JSON from this MLA citation. If 
           any field can not be found, return it as null. Use title, authors, and doi as keys, author's value      
           should be a list of authors. {citation}\n\nCitation JSON:", 'disable_doc_valid_check': False,           
           'defer_embedding': False, 'doc_filters': None, 'use_human_readable_clinical_trials': False,             
           'enrichment_llm': 'selfhost-nemotron-vlm', 'enrichment_llm_config': {'model_list': [{'model_name':      
           'selfhost-nemotron-vlm', 'litellm_params': {'model': 'openai/nvidia/nemotron-nano-12b-v2-vl',           
           'api_base': 'http://localhost:8004/v1', 'api_key': 'dummy', 'temperature': 0, 'max_tokens': 2048}}]},   
           'enrichment_page_radius': 1, 'enrichment_prompt': "You are analyzing an image, formula, or table from a 
           scientific document. Provide a detailed description that will be used to answer questions about its     
           content. Focus on key elements, data, relationships, variables, and scientific insights visible in the  
           image. It's especially important to document referential information such as figure/table numbers,      
           labels, plot colors, or legends.\n\nText co-located with the media may be associated with other media or
           unrelated content, so do not just blindly quo

INFO: Beginning agent 'ToolSelector' run with question 'What experiments are carried out?' and full settings {'llm': 'selfhost-nemotron-vlm', 'llm_config': {'model_list': [{'model_name': 'selfhost-nemotron-vlm', 'litellm_params': {'model': 'openai/nvidia/nemotron-nano-12b-v2-vl', 'api_base': 'http://localhost:8004/v1', 'api_key': 'dummy', 'temperature': 0, 'max_tokens': 2048}}]}, 'summary_llm': 'selfhost-nemotron-vlm', 'summary_llm_config': {'model_list': [{'model_name': 'selfhost-nemotron-vlm', 'litellm_params': {'model': 'openai/nvidia/nemotron-nano-12b-v2-vl', 'api_base': 'http://localhost:8004/v1', 'api_key': 'dummy', 'temperature': 0, 'max_tokens': 2048}}]}, 'embedding': 'openai/nvidia/llama-3.2-nv-embedqa-1b-v2', 'embedding_config': {'kwargs': {'api_base': 'http://localhost:8003/v1', 'api_key': 'dummy', 'encoding_format': 'float', 'input_type': 'passage'}}, 'temperature': 0.0, 'batch_size': 1, 'texts_index_mmr_lambda': 1.0, 'verbosity': 3, 'answer': {'evidence_k': 5, 'evidence_re

           No changes to index.

DEBUG: No changes to index.
INFO: Routing strategy: simple-shuffle


...

           agent_response: session=PQASession(id=UUID('2ae69d78-056e-4afc-90e4-3044fcfb5f26'), question='What      
           experiments are carried out?', answer='The experiments described in the context involve training and    
           evaluating Transformer models for machine translation tasks (English-to-German and English-to-French) on
           the WMT 2014 dataset, with comparisons to prior models. Variations in architecture—such as the number of
           attention heads, key/value dimensions, and dropout rates—are tested to evaluate their impact on BLEU    
           scores and training costs (Vaswani2017 pages 8-9). Additionally, the Transformer architecture is applied
           to English constituency parsing on the Penn Treebank dataset (Vaswani2017 pages 8-9).  \n\nSeparately,  
           studies on acid-sensing ion channel 1a (ASIC1a) in neuromuscular transmission include                   
           electrophysiological recordings to measure synaptic responses, analysis of neuromuscular junction (NMJ) 
           morphology, and functional assessments of muscle fatigue in female mice                                 
           (urbano2014acidsensingionchannels pages 10-10, urbano2014acidsensingionchannels pages 11-11). These     
           experiments demonstrate that ASIC1a inhibits neurotransmitter release, with female mice exhibiting      
           reduced NMJ function and increased fatigue compared to males (urbano2014acidsensingionchannels pages    
           10-10). Further experiments in ASIC1a knockout mice assess the impact of extracellular acidosis on      
           calcium influx via ASIC1a and explore its synaptic localization through interactions with PSD-95        
           (urbano2014acidsensingionchannels pages 11-11). The research also links ASIC1a dysfunction to           
           neurodegenerative and seizure-related outcomes, such as motor neuron degeneration and seizure           
           termination mechanisms (urbano2014acidsensingionchannels pages 11-11).\n', raw_answer='The experiments  
           described in the context involve training and evaluating Transformer models for machine translation     
           tasks (English-to-German and English-to-French) on the WMT 2014 dataset, with comparisons to prior      
           models. Variations in architecture—such as the number of attention heads, key/value dimensions, and     
           dropout rates—are tested to evaluate their impact on BLEU scores and training costs (pqac-ab479f81).    
           Additionally, the Transformer architecture is applied to English constituency parsing on the Penn       
           Treebank dataset (pqac-ab479f81).  \n\nSeparately, studies on acid-sensing ion channel 1a (ASIC1a) in   
           neuromuscular transmission include electrophysiological recordings to measure synaptic responses,       
           analysis of neuromuscular junction (NMJ) morphology, and functional assessments of muscle fatigue in    
           female mice (pqac-be1ccadb, pqac-0008e951). These experiments demonstrate that ASIC1a inhibits          
           neurotransmitter release, with female mice exhibiting reduced NMJ function and increased fatigue        
           compared to males (pqac-be1ccadb). Further experiments in ASIC1a knockout mice assess the impact of     
           extracellular acidosis on calcium influx via ASIC1a and explore its synaptic localization through       
           interactions with PSD-95 (pqac-0008e951). The research also links ASIC1a dysfunction to                 
           neurodegenerative and seizure-related outcomes, such as motor neuron degeneration and seizure           
           termination mechanisms (pqac-0008e951).\n', answer_reasoning=None, has_successful_answer=True,          
           context='pqac-ab479f81: The experiments include training and evaluating Transformer models for machine  
           translation tasks (English-to-German and Engl

DEBUG: agent_response: session=PQASession(id=UUID('2ae69d78-056e-4afc-90e4-3044fcfb5f26'), question='What experiments are carried out?', answer='The experiments described in the context involve training and evaluating Transformer models for machine translation tasks (English-to-German and English-to-French) on the WMT 2014 dataset, with comparisons to prior models. Variations in architecture—such as the number of attention heads, key/value dimensions, and dropout rates—are tested to evaluate their impact on BLEU scores and training costs (Vaswani2017 pages 8-9). Additionally, the Transformer architecture is applied to English constituency parsing on the Penn Treebank dataset (Vaswani2017 pages 8-9).  \n\nSeparately, studies on acid-sensing ion channel 1a (ASIC1a) in neuromuscular transmission include electrophysiological recordings to measure synaptic responses, analysis of neuromuscular junction (NMJ) morphology, and functional assessments of muscle fatigue in female mice (urbano2014a

           Answer: The experiments described in the context involve training and evaluating Transformer models for 
           machine translation tasks (English-to-German and English-to-French) on the WMT 2014 dataset, with       
           comparisons to prior models. Variations in architecture—such as the number of attention heads, key/value
           dimensions, and dropout rates—are tested to evaluate their impact on BLEU scores and training costs     
           (Vaswani2017 pages 8-9). Additionally, the Transformer architecture is applied to English constituency  
           parsing on the Penn Treebank dataset (Vaswani2017 pages 8-9).                                           
                                                                                                                   
           Separately, studies on acid-sensing ion channel 1a (ASIC1a) in neuromuscular transmission include       
           electrophysiological recordings to measure synaptic responses, analysis of neuromuscular junction (NMJ) 
           morphology, and functional assessments of muscle fatigue in female mice                                 
           (urbano2014acidsensingionchannels pages 10-10, urbano2014acidsensingionchannels pages 11-11). These     
           experiments demonstrate that ASIC1a inhibits neurotransmitter release, with female mice exhibiting      
           reduced NMJ function and increased fatigue compared to males (urbano2014acidsensingionchannels pages    
           10-10). Further experiments in ASIC1a knockout mice assess the impact of extracellular acidosis on      
           calcium influx via ASIC1a and explore its synaptic localization through interactions with PSD-95        
           (urbano2014acidsensingionchannels pages 11-11). The research also links ASIC1a dysfunction to           
           neurodegenerative and seizure-related outcomes, such as motor neuron degeneration and seizure           
           termination mechanisms (urbano2014acidsensingionchannels pages 11-11).                                  
           

INFO: [bold blue]Answer: The experiments described in the context involve training and evaluating Transformer models for machine translation tasks (English-to-German and English-to-French) on the WMT 2014 dataset, with comparisons to prior models. Variations in architecture—such as the number of attention heads, key/value dimensions, and dropout rates—are tested to evaluate their impact on BLEU scores and training costs (Vaswani2017 pages 8-9). Additionally, the Transformer architecture is applied to English constituency parsing on the Penn Treebank dataset (Vaswani2017 pages 8-9).  

Separately, studies on acid-sensing ion channel 1a (ASIC1a) in neuromuscular transmission include electrophysiological recordings to measure synaptic responses, analysis of neuromuscular junction (NMJ) morphology, and functional assessments of muscle fatigue in female mice (urbano2014acidsensingionchannels pages 10-10, urbano2014acidsensingionchannels pages 11-11). These experiments demonstrate that ASIC1

✅ Agent query SUCCESS!

--- Status: success ---

Question:
What experiments are carried out?

Answer:
The experiments described in the context involve training and evaluating Transformer models for machine translation tasks (English-to-German and English-to-French) on the WMT 2014 dataset, with comparisons to prior models. Variations in architecture—such as the number of attention heads, key/value dimensions, and dropout rates—are tested to evaluate their impact on BLEU scores and training costs (Vaswani2017 pages 8-9). Additionally, the Transformer architecture is applied to English constituency parsing on the Penn Treebank dataset (Vaswani2017 pages 8-9).  

Separately, studies on acid-sensing ion channel 1a (ASIC1a) in neuromuscular transmission include electrophysiological recordings to measure synaptic responses, analysis of neuromuscular junction (NMJ) morphology, and functional assessments of muscle fatigue in female mice (urbano2014acidsensingionchannels pages 10-10, urbano2014